# HITL模式

Human In The Loop人机协同设计模式，它允许计算图在运行过程中暂停，等待人工审批、修正意见或补充信息，然后从暂停位置恢复执行


In [1]:
import os
from typing import Literal, TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from loguru import logger

_ = load_dotenv(override=True)

In [2]:
llm = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
    temperature=0.7,
)

## 中断

1. 配置检查点存储器

2. 设置 **`thread_id`**

   如果要确保中断后可以恢复执行，就需要保存运行图的完整状态，所以必须**启用可恢复执行**。

3. 在需要中断的位置调用 **`interrupt()`**

In [3]:
class HITLState(TypedDict):
    username: str


def node_a(state: HITLState):
    logger.info("Before: node_a interrupt ")
    username = interrupt("怎么称呼你？")
    logger.info(f"After: node_a interrupt receive {username=}")
    return {"username": username}

In [4]:
graph_builder = StateGraph(state_schema=HITLState)
graph_builder.add_node(node_a)
graph_builder.add_edge(START, "node_a")
graph_builder.add_edge("node_a", END)

graph = graph_builder.compile(checkpointer=InMemorySaver())

In [5]:
config = {"configurable": {"thread_id": "20261021-1"}}

In [6]:
result = graph.invoke({}, config=config)

2026-10-21 16:16:00.433 | INFO     | __main__:node_a:6 - Before: node_a interrupt 


In [7]:
result

{'__interrupt__': [Interrupt(value='怎么称呼你？', id='42432eab4c4faaa942bfbceef4858974', response_schema=None)]}

## 恢复

满足以下要求：

- 基于相同的配置再次调用计算图

- 将输入替换为 **`Command()`** 实例即可恢复运行。

**`LangGraph`** 会从中断节点继续运行，该节点会被再次执行。

通过 **`Command`** 实例的 **`resume`** 属性将用户反馈传递给计算图，中断节点重新运行时，传递给 **`resume`** 属性的值将会作为 **`interrupt()`** 函数的返回值。

In [8]:
msg = result["__interrupt__"][0].value
print(msg)
# 模拟用户手动输入
user_input = "鲨鱼不吃辣椒"

怎么称呼你？


In [9]:
res = graph.invoke(Command(resume=user_input), config=config)
res

2026-10-21 16:16:00.512 | INFO     | __main__:node_a:6 - Before: node_a interrupt 
2026-10-21 16:16:00.514 | INFO     | __main__:node_a:8 - After: node_a interrupt receive username='鲨鱼不吃辣椒'


{'username': '鲨鱼不吃辣椒'}

# 并行中断

In [10]:
class ParallState(TypedDict):
    username: str
    age: int


def node_b(state: HITLState):
    logger.info("Before: node_b interrupt ")
    age = interrupt("请输入您的年龄")
    logger.info(f"After: node_b interrupt recevice {age=}")
    return {"age": int(age)}


graph_builder = StateGraph(state_schema=ParallState)
graph_builder.add_node("node_a", node_a)
graph_builder.add_node("node_b", node_b)
graph_builder.add_edge(START, "node_a")
graph_builder.add_edge(START, "node_b")
graph_builder.add_edge("node_a", END)
graph_builder.add_edge("node_b", END)

graph = graph_builder.compile(checkpointer=InMemorySaver())

In [11]:
res = graph.invoke({}, config=config)

2026-10-21 16:16:00.563 | INFO     | __main__:node_a:6 - Before: node_a interrupt 
2026-10-21 16:16:00.567 | INFO     | __main__:node_b:7 - Before: node_b interrupt 


In [12]:
res

{'__interrupt__': [Interrupt(value='怎么称呼你？', id='48ef15ab7fa1f536c8623403151e6d1c', response_schema=None),
  Interrupt(value='请输入您的年龄', id='2e6bbcd2d33153914bff8b11d796c37f', response_schema=None)]}

恢复，根据id进行恢复

In [13]:
resume_map = {}
for item in res["__interrupt__"]:
    if "称呼" in item.value:
        resume_map[item.id] = "鲨鱼不吃辣椒"
    else:
        resume_map[item.id] = 18
resume_map

{'48ef15ab7fa1f536c8623403151e6d1c': '鲨鱼不吃辣椒',
 '2e6bbcd2d33153914bff8b11d796c37f': 18}

In [14]:
result = graph.invoke(Command(resume=resume_map), config=config)

2026-10-21 16:16:00.627 | INFO     | __main__:node_a:6 - Before: node_a interrupt 
2026-10-21 16:16:00.632 | INFO     | __main__:node_a:8 - After: node_a interrupt receive username='鲨鱼不吃辣椒'
2026-10-21 16:16:00.631 | INFO     | __main__:node_b:7 - Before: node_b interrupt 
2026-10-21 16:16:00.648 | INFO     | __main__:node_b:9 - After: node_b interrupt recevice age=18


In [15]:
result

{'username': '鲨鱼不吃辣椒', 'age': 18}

# 审批模式

In [16]:
class ApproveState(TypedDict):
    topic: str
    result: str
    is_approved: bool


def approve_node(state: ApproveState) -> Command[Literal["llm_node", "fallback_node"]]:
    """添加返回的类型，用于langgraph生成图"""
    logger.info("执行approve_node")
    is_approved = interrupt("是否同意调用大模型")
    logger.info("授权成功😃" if is_approved else "授权失败😭")
    goto = "llm_node" if is_approved else "fallback_node"
    return Command(update={"is_approved": is_approved}, goto=goto)


def llm_node(state: ApproveState):
    logger.info("执行llm_node")
    ai_msg = llm.invoke(f"列举关于{state['topic']}的三个优点，添加合适的emoji")
    return {"result": ai_msg.content}


def fallback_node(state: ApproveState):
    logger.info("执行fallback_node")
    return {"result": "请求被拒绝"}

In [17]:
graph_builder = StateGraph(state_schema=ApproveState)

graph_builder.add_node(approve_node).add_node(llm_node).add_node(fallback_node)
graph_builder.add_edge(START, "approve_node")
graph_builder.add_edge("llm_node", END)
graph_builder.add_edge("fallback_node", END)

graph = graph_builder.compile(checkpointer=InMemorySaver())

In [18]:
print(graph.get_graph().draw_ascii())

               +-----------+              
               | __start__ |              
               +-----------+              
                     *                    
                     *                    
                     *                    
             +--------------+             
             | approve_node |             
             +--------------+             
              ...          ..             
             .               ..           
           ..                  ..         
+---------------+           +----------+  
| fallback_node |           | llm_node |  
+---------------+           +----------+  
              ***          **             
                 *       **               
                  **   **                 
                +---------+               
                | __end__ |               
                +---------+               


In [ ]:
config = {"configurable": {"thread_id": "approve-chat-1"}}
result = graph.invoke({"topic": "Java"}, config=config)

2026-10-21 16:16:00.894 | INFO     | __main__:approve_node:9 - 执行approve_node


In [21]:
print(result)

{'topic': 'Java', '__interrupt__': [Interrupt(value='是否同意调用大模型', id='e275050bf2febecaaf0f70981407bb26', response_schema=None)]}


In [22]:
if "__interrupt__" in result:
    print(result["__interrupt__"][0].value)
    user_answer = "YES"
    # 恢复调用
    result = graph.invoke(Command(resume=(user_answer == "YES")), config=config)
result

2026-10-21 16:16:00.921 | INFO     | __main__:approve_node:9 - 执行approve_node
2026-10-21 16:16:00.921 | INFO     | __main__:approve_node:11 - 授权成功😃
2026-10-21 16:16:00.923 | INFO     | __main__:llm_node:17 - 执行llm_node


是否同意调用大模型


{'topic': 'Java',
 'result': '1. 🌍 **跨平台性强**：基于 JVM，实现“一次编写，到处运行”，方便部署到不同系统。  \n2. 🧰 **生态成熟**：拥有丰富的框架、类库和开发工具，企业级开发支持完善。  \n3. 🛡️ **稳定健壮**：强类型语言、自动垃圾回收机制，加上长期广泛使用，使程序更可靠、易维护。',
 'is_approved': True}

演示授权失败

In [ ]:
config = {"configurable": {"thread_id": "approve-chat-2"}}
result = graph.invoke({"topic": "Python"}, config=config)
result

2026-10-21 16:16:05.115 | INFO     | __main__:approve_node:9 - 执行approve_node


{'topic': 'Python',
 'result': '',
 'is_approved': True,
 '__interrupt__': [Interrupt(value='是否同意调用大模型', id='db03423529147eb66baf52c91b4d5404', response_schema=None)]}

In [24]:
if "__interrupt__" in result:
    print(result["__interrupt__"][0].value)
    # 这次不同意
    user_answer = "NO"
    # 恢复调用
    result = graph.invoke(Command(resume=(user_answer == "YES")), config=config)
result

2026-10-21 16:16:05.132 | INFO     | __main__:approve_node:9 - 执行approve_node
2026-10-21 16:16:05.133 | INFO     | __main__:approve_node:11 - 授权失败😭
2026-10-21 16:16:05.136 | INFO     | __main__:fallback_node:23 - 执行fallback_node


是否同意调用大模型


{'topic': 'Python', 'result': '请求被拒绝', 'is_approved': False}